# Not your grandmother’s toolbox– the Robotics Toolbox reinvented for python
### Peter Corke and Jesse Haviland

This is the code for the examples in the paper published at ICRA2021.


In [ ]:
# RTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_rtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for roboticstoolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed() -> bool:
    """Install roboticstoolbox-python if needed, and report the environment.

    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        # spatialmath-python <= 1.1.18 lists pre-commit (a dev tool) as a runtime
        # dependency; its virtualenv requires packaging>=26.3, which conflicts with
        # the packaging 24.2 built into Pyodide, so a normal resolve fails.  Install
        # it without dependencies first: it is then already satisfied when the
        # toolbox's own dependencies are resolved below.  Remove this line once a
        # spatialmath release without that dependency is out
        # (rai-opensource/spatialmath-python#250).
        await micropip.install("spatialmath-python", deps=False)
        await micropip.install("ipympl")

        wheels = sorted(Path("/pypi").glob("roboticstoolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site. Relative,
            # not absolute: a leading slash resolves against the origin, not
            # this site's own base URL (a GitHub Pages project subpath).
            await micropip.install(f"pypi/{wheels[-1].name}")
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("roboticstoolbox-python")
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
        else:
            print("Installing roboticstoolbox-python...")
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "roboticstoolbox-python",
                ],
                check=True,
            )
            where, colab = "on Colab", True

    import roboticstoolbox

    version = getattr(roboticstoolbox, "__version__", "unknown")
    print(f"Running {where} using RTB v{version}")
    return colab

COLAB = await ensure_installed()


In [ ]:
from math import pi
import numpy as np

# display result of assignments
%config ZMQInteractiveShell.ast_node_interactivity = 'last_expr_or_assign'
# make NumPy display a bit nicer
np.set_printoptions(
    linewidth=100,
    formatter={"float": lambda x: f"{x:10.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"},
)
# make cells nice and wide
from IPython.display import display, HTML

display(
    HTML(
        data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""
    )
)
%matplotlib ipympl

# III.SPATIAL MATHEMATICS

In [ ]:
from spatialmath.base import *

T = transl(0.5, 0.0, 0.0) @ rpy2tr(0.1, 0.2, 0.3, order="xyz") @ trotx(-90, "deg")

In [ ]:
from spatialmath import *

T = SE3(0.5, 0.0, 0.0) * SE3.RPY([0.1, 0.2, 0.3], order="xyz") * SE3.Rx(-90, unit="deg")

In [ ]:
T.eul()

In [ ]:
T.R

In [ ]:
T.plot(color="red", label="2")

In [ ]:
UnitQuaternion.Rx(0.3)
UnitQuaternion.AngVec(0.3, [1, 0, 0])

In [ ]:
R = SE3.Rx(np.linspace(0, pi / 2, num=100))
len(R)

# IV. ROBOTICS TOOLBOX
## A. Robot models

In [ ]:
from roboticstoolbox import *

# robot length values (metres)
d1 = 0.352
a1 = 0.070
a2 = 0.360
d4 = 0.380
d6 = 0.065;

In [ ]:
robot = DHRobot(
    [
        RevoluteDH(d=d1, a=a1, alpha=-pi / 2),
        RevoluteDH(a=a2),
        RevoluteDH(alpha=pi / 2),
    ],
    name="my IRB140",
)

In [ ]:
puma = models.DH.Puma560()

In [ ]:
T = puma.fkine([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])

In [ ]:
sol = puma.ikine_LM(T)

In [ ]:
puma.plot(sol.q);

In [ ]:
puma.ikine_a(T, config="lun")

In [ ]:
from roboticstoolbox import ETS as ET

In [ ]:
# Puma dimensions (m), see RVC2 Fig. 7.4 for details
l1 = 0.672
l2 = -0.2337
l3 = 0.4318
l4 = 0.0203
l5 = 0.0837
l6 = 0.4318;

In [ ]:
e = (
    ET.tz(l1)
    * ET.rz()
    * ET.ty(l2)
    * ET.ry()
    * ET.tz(l3)
    * ET.tx(l4)
    * ET.ty(l5)
    * ET.ry()
    * ET.tz(l6)
    * ET.rz()
    * ET.ry()
    * ET.rz()
)

In [ ]:
robot = ERobot(e)
print(robot)

In [ ]:
panda = models.URDF.Panda()
print(panda)

## B. Trajectories

In [ ]:
traj = jtraj(puma.qz, puma.qr, 100)
traj.qplot()

In [ ]:
t = np.arange(0, 2, 0.010)
T0 = SE3(0.6, -0.5, 0.3)

In [ ]:
T1 = SE3(0.4, 0.5, 0.2)

In [ ]:
Ts = ctraj(T0, T1, t)
len(Ts)

In [ ]:
sol = puma.ikine_LM(Ts)
sol.q.shape

## C. Symbolic manipulation

In [ ]:
import spatialmath.base.symbolic as sym

phi, theta, psi = sym.symbol("φ, ϴ, ψ")
rpy2r(phi, theta, psi)

In [ ]:
q = sym.symbol("q_:6")  # q = (q_1, q_2, ... q_5)
T = puma.fkine(q);

In [ ]:
puma = models.DH.Puma560(symbolic=True)
T = puma.fkine(q)
T.t[0]

In [ ]:
puma = models.DH.Puma560(symbolic=False)
J = puma.jacob0(puma.qn)

In [ ]:
J = puma.jacobe(puma.qn)

## D. Differential kinematics

In [ ]:
J = puma.jacob0(puma.qr)

In [ ]:
np.linalg.matrix_rank(J)

In [ ]:
jsingu(J)

In [ ]:
H = panda.hessian0(panda.qz)
H.shape

In [ ]:
puma.manipulability(puma.qn)

In [ ]:
puma.manipulability(puma.qn, method="asada")

In [ ]:
puma.manipulability(puma.qn, axes="trans")

In [ ]:
panda.jacobm(panda.qr)

## E. Dynamics

In [ ]:
tau = puma.rne(puma.qn, np.zeros((6,)), np.zeros((6,)))

In [ ]:
J = puma.inertia(puma.qn)

In [ ]:
C = puma.coriolis(puma.qn, 0.1 * np.ones((6,)))

In [ ]:
g = puma.gravload(puma.qn)

In [ ]:
qdd = puma.accel(puma.qn, tau, np.zeros((6,)))

# V. NEW CAPABILITY
## B. Collision checking

In [ ]:
from spatialgeometry import Box

obstacle = Box([1, 1, 1], base=SE3(1, 0, 0))
iscollision = panda.collided(panda.qr, obstacle)  # boolean
iscollision = panda.links[0].collided(obstacle)

In [ ]:
d, p1, p2 = panda.closest_point(panda.qr, obstacle)
print(d, p1, p2)
d, p1, p2 = panda.links[0].closest_point(obstacle)
print(d, p1, p2)

## C. Interfaces

In [ ]:
panda.plot(panda.qr);

In [ ]:
from roboticstoolbox.backends.swift import Swift

backend = Swift()
backend.launch()  # create graphical world
backend.add(panda)  # add robot to the world
panda.q = panda.qr  # update the robot
backend.step()  # display the world